# Logical Fallacy Detection
**Dataset:** CoCoLoFa (Comments with Common Logical Fallacies)
https://github.com/Crowd-AI-Lab/cocolofa

**Citation:**
Min-Hsuan Yeh, Ruyuan Wan, and Ting-Hao 'Kenneth' Huang. (2024). *CoCoLoFa: A Dataset of News Comments with Common Logical Fallacies Written by LLM-Assisted Crowds*. [arXiv:2410.03457](https://arxiv.org/abs/2410.03457)

## 0. Import Libraries

In [1]:
import json
import re
import os
import numpy as np
import pandas as pd

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

## 1. Load Dataset

In [2]:
DATA_DIR = 'data'   

file_path = os.path.join(DATA_DIR, 'train.json')

if not os.path.exists(file_path):
    print(f"Error: Could not find train.json at {file_path}")
else:
    with open(file_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
        
    records = []
    for article in raw_data:
        for comment in article['comments']:
            records.append({
                'comment_id': comment['id'],
                'news_id': comment['news_id'],
                'fallacy': comment['fallacy'],
                'comment': comment['comment']
            })

    df = pd.DataFrame(records)

    print(f"Comments amount: {len(df)}")
    print(f"\nFallacy Label Distribution:")
    print(df['fallacy'].value_counts())
    
    display(df.head(10))

Comments amount: 5370

Fallacy Label Distribution:
fallacy
none                        2202
slippery slope               431
appeal to worse problems     421
appeal to nature             412
appeal to tradition          401
false dilemma                391
appeal to majority           383
hasty generalization         379
appeal to authority          350
Name: count, dtype: int64


,comment_id,news_id,fallacy,comment
0,5584,262,none,Lack of transparency in government isn't unexp...
1,5582,262,appeal to authority,While the issues discussed here should be addr...
2,5583,262,none,The excuse that Brazilian municipalities do no...
3,6183,262,none,This is what's to be expected of developing an...
4,6182,262,appeal to tradition,"Sad to say, I have to agree with you. Rulers c..."
5,6184,262,none,Why doesn't the Brazilian federal government s...
6,6782,262,appeal to worse problems,At least they have a set up for it. In some co...
7,6783,262,hasty generalization,If these people can't even manage doing someth...
8,6784,262,appeal to tradition,I completely agree with you! There have alway...
9,10082,262,slippery slope,The argument that people should have access to...


## 2. Text Cleaning

In [3]:
def clean_text(text):
    text = text.lower()                                   
    text = re.sub(r'http\S+|www\.\S+', '', text)      
    text = re.sub(r'<[^>]+>', '', text)                 
    text = re.sub(r'@\w+', '', text)                     
    text = re.sub(r'#\w+', '', text)                
    text = re.sub(r'[^a-zA-Z\s]', '', text)              
    text = re.sub(r'\s+', ' ', text).strip()            
    return text

df['cleaned'] = df['comment'].apply(clean_text)

print("Contoh sebelum & sesudah cleaning:")
for i in range(3):
    print(f"\n[BEFORE]: {df['comment'].iloc[i][:120]}...")
    print(f"[AFTER ]: {df['cleaned'].iloc[i][:120]}...")

Contoh sebelum & sesudah cleaning:

[BEFORE]: Lack of transparency in government isn't unexpected.  No one likes to be under scrutiny,  especially not those with powe...
[AFTER ]: lack of transparency in government isnt unexpected no one likes to be under scrutiny especially not those with power tra...

[BEFORE]: While the issues discussed here should be addressed, there need to be priorities. The authorities of the country probabl...
[AFTER ]: while the issues discussed here should be addressed there need to be priorities the authorities of the country probably ...

[BEFORE]: The excuse that Brazilian municipalities do not have the funds to handle the public service requests does not hold water...
[AFTER ]: the excuse that brazilian municipalities do not have the funds to handle the public service requests does not hold water...


## 3. Tokenization

In [4]:
df['tokens'] = df['cleaned'].apply(word_tokenize)

print("Contoh hasil tokenisasi:")
for i in range(3):
    print(f"\n[CLEANED ]: {df['cleaned'].iloc[i][:100]}...")
    print(f"[TOKENS  ]: {df['tokens'].iloc[i][:10]}...")
    print(f"Jumlah token: {len(df['tokens'].iloc[i])}")

Contoh hasil tokenisasi:

[CLEANED ]: lack of transparency in government isnt unexpected no one likes to be under scrutiny especially not ...
[TOKENS  ]: ['lack', 'of', 'transparency', 'in', 'government', 'isnt', 'unexpected', 'no', 'one', 'likes']...
Jumlah token: 64

[CLEANED ]: while the issues discussed here should be addressed there need to be priorities the authorities of t...
[TOKENS  ]: ['while', 'the', 'issues', 'discussed', 'here', 'should', 'be', 'addressed', 'there', 'need']...
Jumlah token: 55

[CLEANED ]: the excuse that brazilian municipalities do not have the funds to handle the public service requests...
[TOKENS  ]: ['the', 'excuse', 'that', 'brazilian', 'municipalities', 'do', 'not', 'have', 'the', 'funds']...
Jumlah token: 93


## 4. Stopword Removal & Lemmatization

> Dipilih **lemmatization** daripada stemming karena menghasilkan kata dasar yang valid secara bahasa, lebih cocok untuk tugas deteksi fallacy yang membutuhkan pemahaman makna.

In [5]:
fallacy_indicators = {
    'everyone', 'always', 'never', 'must', 'should', 'all', 'every',
    'only', 'just', 'no', 'nothing', 'everything', 'most', 'any',
    'either', 'or', 'neither', 'nor', 'both', 'very', 'too',
}
stop_words = set(stopwords.words('english')) - fallacy_indicators
print(f"Stopwords: {len(set(stopwords.words('english')))} total → {len(stop_words)} after keeping {len(fallacy_indicators)} fallacy indicators")

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [
        lemmatizer.lemmatize(token)
        for token in tokens
        if (token not in stop_words and len(token) > 2) or token in fallacy_indicators
    ]

df['lemmatized'] = df['tokens'].apply(lemmatize_tokens)

# Gabungkan kembali menjadi string untuk keperluan TF-IDF
df['processed_text'] = df['lemmatized'].apply(lambda x: ' '.join(x))

print("Contoh hasil lemmatization:")
for i in range(3):
    print(f"\n[TOKENS     ]: {df['tokens'].iloc[i][:10]}...")
    print(f"[LEMMATIZED ]: {df['lemmatized'].iloc[i][:10]}...")
    print(f"Jumlah token sebelum: {len(df['tokens'].iloc[i])} → sesudah: {len(df['lemmatized'].iloc[i])}")

Stopwords: 198 total → 186 after keeping 21 fallacy indicators
Contoh hasil lemmatization:

[TOKENS     ]: ['lack', 'of', 'transparency', 'in', 'government', 'isnt', 'unexpected', 'no', 'one', 'likes']...
[LEMMATIZED ]: ['lack', 'transparency', 'government', 'isnt', 'unexpected', 'no', 'one', 'like', 'scrutiny', 'especially']...
Jumlah token sebelum: 64 → sesudah: 33

[TOKENS     ]: ['while', 'the', 'issues', 'discussed', 'here', 'should', 'be', 'addressed', 'there', 'need']...
[LEMMATIZED ]: ['issue', 'discussed', 'should', 'addressed', 'need', 'priority', 'authority', 'country', 'probably', 'know']...
Jumlah token sebelum: 55 → sesudah: 31

[TOKENS     ]: ['the', 'excuse', 'that', 'brazilian', 'municipalities', 'do', 'not', 'have', 'the', 'funds']...
[LEMMATIZED ]: ['excuse', 'brazilian', 'municipality', 'fund', 'handle', 'public', 'service', 'request', 'hold', 'water']...
Jumlah token sebelum: 93 → sesudah: 49


## 5. Representasi Teks

### a. TF-IDF (Term Frequency–Inverse Document Frequency) 

In [30]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} dokumen, {tfidf_matrix.shape[1]} fitur")

feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\nContoh 10 fitur (kata/bigram/trigram): {list(feature_names[:10])}")

sum_tfidf = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
top_10_indices = sum_tfidf.argsort()[-10:][::-1]
print("\nTop 10 Fitur")

for idx in top_10_indices:
    print(f"  {feature_names[idx]}: {sum_tfidf[idx]:.4f}")

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names
)

print(f"\nPreview TF-IDF Matrix (5 dokumen pertama, 10 fitur pertama)")
tfidf_df.iloc[:5, :10]

TF-IDF Matrix Shape: (5370, 8000)
  → 5370 dokumen, 8000 fitur

Contoh 10 fitur (kata/bigram/trigram): ['abandon', 'abandoned', 'abhorrent', 'ability', 'able', 'able access', 'able continue', 'able express', 'able get', 'able make']

Top 10 Fitur
  people: 172.0725
  should: 126.9180
  government: 114.5398
  just: 112.2505
  need: 111.8246
  like: 111.5762
  all: 111.5395
  country: 110.7438
  or: 98.6864
  think: 91.0694

Preview TF-IDF Matrix (5 dokumen pertama, 10 fitur pertama)


,abandon,abandoned,abhorrent,ability,able,able access,able continue,able express,able get,able make
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### b. Word2Vec
**Word2Vec** mempelajari embedding vektor untuk setiap kata berdasarkan konteks kemunculannya.
- Menangkap hubungan semantik antar kata (misal: "authority" dekat dengan "expert", "tradition" dekat dengan "culture").
- Representasi dokumen diperoleh dengan merata-ratakan vektor semua kata dalam dokumen.
- Menghasilkan dense vector berdimensi tetap untuk setiap dokumen.

In [27]:
w2v_model = Word2Vec(
    sentences=df['lemmatized'].tolist(),
    vector_size=100,    
    window=5,           
    min_count=2,    
    workers=4,
    epochs=20,
    sg=1            
)

print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector dimension: {w2v_model.wv.vector_size}")

print(f"\nContoh vektor untuk kata 'government':")
print(w2v_model.wv['government'])

for word in ['government', 'freedom', 'tradition']:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f"\nKata mirip dengan '{word}':")
        for w, score in similar:
            print(f"  {w}: {score:.4f}")

Vocabulary size: 6654
Vector dimension: 100

Contoh vektor untuk kata 'government':
[-1.09370835e-01  8.91975611e-02  1.99608412e-02 -2.35377356e-01
 -2.19719694e-03 -2.11431488e-01  2.33695030e-01  5.56663692e-01
 -1.12695880e-01  4.14498672e-02 -1.52545482e-01 -4.63700086e-01
 -8.68575051e-02 -4.81344797e-02 -3.12110752e-01 -4.79480743e-01
  7.95075372e-02  1.32144719e-01  4.78748716e-02 -7.74997830e-01
  3.37389112e-01  3.12986791e-01  5.57942629e-01 -1.86335176e-01
 -1.15891367e-01 -5.16124032e-02 -1.53943285e-01 -1.08432181e-01
 -3.03832293e-01 -5.86738363e-02 -2.46487521e-02  1.60157293e-01
 -6.54333038e-03 -3.33193064e-01  3.59013118e-02  1.99772745e-01
  2.51602709e-01  7.97760859e-02 -2.55187601e-01 -2.53493667e-01
  3.20952535e-01 -2.22417310e-01  1.76105499e-01  1.10391341e-01
 -7.65912682e-02 -1.47677526e-01 -3.97071481e-01  2.30295002e-01
 -2.66222626e-01 -1.80656835e-02  5.19435108e-01 -4.24922891e-02
  3.90248269e-01 -2.13011459e-01  1.32367179e-01  2.99151927e-01
  2.39

In [12]:
def document_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

w2v_vectors = np.array([
    document_vector(tokens, w2v_model) for tokens in df['lemmatized']
])

w2v_df = pd.DataFrame(
    w2v_vectors,
    columns=[f'w2v_dim_{i}' for i in range(w2v_model.wv.vector_size)]
)

print(f"Word2Vec Document Matrix Shape: {w2v_vectors.shape}")
print(f"  → {w2v_vectors.shape[0]} dokumen, {w2v_vectors.shape[1]} dimensi")
print(f"Preview Word2Vec Document Vectors (5 dokumen pertama, 10 dimensi pertama)")
w2v_df.iloc[:5, :10]

Word2Vec Document Matrix Shape: (5370, 100)
  → 5370 dokumen, 100 dimensi
Preview Word2Vec Document Vectors (5 dokumen pertama, 10 dimensi pertama)


,w2v_dim_0,w2v_dim_1,w2v_dim_2,w2v_dim_3,w2v_dim_4,w2v_dim_5,w2v_dim_6,w2v_dim_7,w2v_dim_8,w2v_dim_9
0,-0.184890,0.269799,-0.036520,0.063788,0.064982,-0.192580,0.215105,0.369354,-0.150526,-0.047355
1,-0.124267,0.155330,0.016827,0.220654,0.041431,-0.258398,0.120884,0.447194,-0.196653,-0.185523
2,-0.342258,0.236241,-0.028594,0.056062,0.187976,-0.259403,0.113392,0.448578,-0.142079,0.062972
3,-0.094386,0.127090,0.032672,0.069615,0.086422,-0.236575,0.024291,0.462124,-0.128991,-0.118400
4,-0.186062,0.239861,-0.168576,0.027615,0.139514,-0.283479,0.083053,0.500204,-0.205486,-0.093159
